# ResNet18 FER2013 Experiments

Focused transfer-learning notebook for ResNet18. Checkpoints are selected by validation macro F1, with validation loss used as the tie-breaker.

In [1]:
from pathlib import Path
import json
import random
import sys

import importlib
import numpy as np
import pandas as pd
import torch

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

DATA_DIR = PROJECT_ROOT / 'data' / 'raw' / 'fer2013_images'
# Kaggle option:
# DATA_DIR = Path('/kaggle/input/datasets/msambare/fer2013')

RESULTS_DIR = PROJECT_ROOT / 'results'
(RESULTS_DIR / 'checkpoints').mkdir(parents=True, exist_ok=True)
(RESULTS_DIR / 'metrics').mkdir(parents=True, exist_ok=True)
(RESULTS_DIR / 'figures').mkdir(parents=True, exist_ok=True)

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

SEED = 42
set_seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DEVICE, DATA_DIR

(device(type='cuda'), PosixPath('/kaggle/working/data/raw/fer2013_images'))

## Kaggle Path Setup Option

Use this block only when running the notebook in Kaggle. It clones or updates the repository, then points `DATA_DIR` to the Kaggle FER2013 dataset folder.

In [3]:
# Kaggle option. Uncomment this block only when running in Kaggle.

%cd /kaggle/working
from pathlib import Path
import sys

REPO_URL = 'https://github.com/zsykk/DL-final-project.git'
REPO_DIR = 'DL-final-project'

if Path(REPO_DIR).exists():
    %cd /kaggle/working/DL-final-project
    !git pull
else:
    !git clone {REPO_URL}
    %cd /kaggle/working/DL-final-project

PROJECT_ROOT = Path('/kaggle/working/DL-final-project')
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
DATA_DIR = Path('/kaggle/input/datasets/msambare/fer2013')
RESULTS_DIR = PROJECT_ROOT / 'results'
(RESULTS_DIR / 'checkpoints').mkdir(parents=True, exist_ok=True)
(RESULTS_DIR / 'metrics').mkdir(parents=True, exist_ok=True)
(RESULTS_DIR / 'figures').mkdir(parents=True, exist_ok=True)

DATA_DIR, (DATA_DIR / 'train').exists(), (DATA_DIR / 'test').exists(), PROJECT_ROOT

/kaggle/working
/kaggle/working/DL-final-project
Already up to date.


(PosixPath('/kaggle/input/datasets/msambare/fer2013'),
 True,
 True,
 PosixPath('/kaggle/working/DL-final-project'))

In [ ]:
from fer_project.data import TransformConfig, build_imagefolder_dataloaders, class_weights, dataset_labels
from fer_project.models import build_model, set_resnet18_trainable_layers

import fer_project.training as training
training = importlib.reload(training)
fit = training.fit
fit_sgd_schedule = training.fit_sgd_schedule

import fer_project.metrics as metrics
metrics = importlib.reload(metrics)
collect_predictions = metrics.collect_predictions
plot_training_and_confusion = metrics.plot_training_and_confusion
save_classification_report = metrics.save_classification_report
top_confusions = metrics.top_confusions


In [ ]:
TRANSFER_TRAIN_CONFIG = TransformConfig(image_size=224, channels=3, augment=True, imagenet_norm=True)
TRANSFER_EVAL_CONFIG = TransformConfig(image_size=224, channels=3, augment=False, imagenet_norm=True)
RESNET18_SMALL_CLASSIFIER = [256]
RESNET18_CLASSIFIER_DROPOUT = 0.4

EMOTION_ROWS = ['angry', 'disgust', 'fear', 'happy', 'sad', 'surprise', 'neutral']


def save_experiment_config(name, config):
    path = RESULTS_DIR / 'metrics' / f'{name}_config.json'
    with open(path, 'w') as f:
        json.dump(config, f, indent=2, default=str)
    return path


def build_transfer_loaders(batch_size=64, num_workers=0, subset_fraction=1.0, weighted_sampler=False):
    return build_imagefolder_dataloaders(
        DATA_DIR,
        train_config=TRANSFER_TRAIN_CONFIG,
        eval_config=TRANSFER_EVAL_CONFIG,
        batch_size=batch_size,
        num_workers=num_workers,
        weighted_sampler=weighted_sampler,
        val_fraction=0.1,
        subset_fraction=subset_fraction,
        seed=SEED,
    )


def best_history_row(history_frame):
    return history_frame.sort_values(['val_macro_f1', 'val_loss'], ascending=[False, True]).iloc[0]


def finalize_model_run(model, loaders, history, name, checkpoint_path):
    model.load_state_dict(torch.load(checkpoint_path, map_location=DEVICE))
    y_true, y_pred = collect_predictions(model, loaders['test'], DEVICE)
    report = save_classification_report(y_true, y_pred, RESULTS_DIR / 'metrics' / f'{name}_classification_report.csv')
    confusions = top_confusions(y_true, y_pred, top_n=10)
    confusions.to_csv(RESULTS_DIR / 'metrics' / f'{name}_top_confusions.csv', index=False)
    history_frame = pd.DataFrame(history)
    history_frame.to_csv(RESULTS_DIR / 'metrics' / f'{name}_history.csv', index=False)
    plot_training_and_confusion(
        history_frame,
        y_true,
        y_pred,
        f'{name}: loss and confusion matrix',
        RESULTS_DIR / 'figures' / f'{name}_confusion_matrix.png',
    )

    best_row = best_history_row(history_frame)
    print(f'Best checkpoint saved to: {checkpoint_path}')
    print('Best validation row:')
    display(best_row[['epoch', 'stage', 'stage_epoch', 'lr', 'train_macro_f1', 'val_macro_f1', 'val_loss']])
    print('Recent training history:')
    display(history_frame.tail())
    print('Test summary:')
    display(report.loc[['macro avg', 'weighted avg']])
    display(report.loc[EMOTION_ROWS, ['precision', 'recall', 'f1-score', 'support']])
    display(confusions)
    return history_frame, report, confusions


## Independent Unfreeze Depth Search

Each cell below starts from a fresh ImageNet-pretrained ResNet18, replaces the classifier with the same dropout-regularized head, freezes the requested prefix of the backbone, and saves its own best checkpoint. This avoids carrying weights from one unfreeze depth into the next run.


In [ ]:
RESNET18_DEPTH_LRS = {
    'fc': 1e-3,
    'layer4': 5e-4,
    'layer3': 1e-4,
    'layer2': 5e-5,
    'layer1': 2e-5,
    'all': 1e-5,
}

DEPTH_SEARCH_NAMES = [
    'resnet18_feature_extract_fc',
    'resnet18_unfreeze_layer4',
    'resnet18_unfreeze_layer3',
    'resnet18_unfreeze_layer2',
]


def build_fresh_resnet18(unfreeze_from='fc', classifier_hidden_layers=RESNET18_SMALL_CLASSIFIER, classifier_dropout=RESNET18_CLASSIFIER_DROPOUT):
    set_seed(SEED)
    model = build_model(
        'transfer',
        transfer_model='resnet18',
        pretrained=True,
        freeze_backbone=False,
        classifier_hidden_layers=classifier_hidden_layers,
        classifier_dropout=classifier_dropout,
    )
    set_resnet18_trainable_layers(model, unfreeze_from)
    return model


def run_resnet18_fresh_experiment(
    name,
    unfreeze_from='fc',
    batch_size=64,
    num_workers=0,
    subset_fraction=1.0,
    epochs=15,
    lr=None,
    optimizer_name='adamw',
    weight_decay=1e-4,
    use_class_weights=False,
    weighted_sampler=False,
    loss_name='cross_entropy',
    focal_gamma=1.0,
    classifier_hidden_layers=RESNET18_SMALL_CLASSIFIER,
    classifier_dropout=RESNET18_CLASSIFIER_DROPOUT,
    lr_decay_rate=0.1,
    lr_plateau_patience=5,
    lr_plateau_threshold=1e-3,
    min_lr=1e-6,
):
    lr = RESNET18_DEPTH_LRS[unfreeze_from] if lr is None else lr
    loaders, datasets = build_transfer_loaders(
        batch_size=batch_size,
        num_workers=num_workers,
        subset_fraction=subset_fraction,
        weighted_sampler=weighted_sampler,
    )
    model = build_fresh_resnet18(
        unfreeze_from=unfreeze_from,
        classifier_hidden_layers=classifier_hidden_layers,
        classifier_dropout=classifier_dropout,
    )
    weights = class_weights(dataset_labels(datasets['train'])) if use_class_weights else None
    checkpoint_path = RESULTS_DIR / 'checkpoints' / f'{name}.pt'

    if optimizer_name == 'sgd_plateau':
        history, _ = fit_sgd_schedule(
            model,
            loaders,
            device=DEVICE,
            epochs=epochs,
            lr=lr,
            weight_decay=weight_decay,
            class_weight=weights,
            loss_name=loss_name,
            focal_gamma=focal_gamma,
            lr_decay_rate=lr_decay_rate,
            lr_plateau_patience=lr_plateau_patience,
            lr_plateau_threshold=lr_plateau_threshold,
            min_lr=min_lr,
            checkpoint_path=checkpoint_path,
            stage_name=f'unfreeze_{unfreeze_from}',
        )
    elif optimizer_name == 'adamw':
        history = fit(
            model,
            loaders,
            device=DEVICE,
            epochs=epochs,
            lr=lr,
            weight_decay=weight_decay,
            class_weight=weights,
            loss_name=loss_name,
            focal_gamma=focal_gamma,
            checkpoint_path=checkpoint_path,
        )
        history = [
            {
                'stage': f'unfreeze_{unfreeze_from}',
                'stage_epoch': row['epoch'],
                'lr': lr,
                **row,
            }
            for row in history
        ]
    else:
        raise ValueError("optimizer_name must be 'adamw' or 'sgd_plateau'")

    save_experiment_config(name, {
        'name': name,
        'model': 'resnet18',
        'source_weights': 'ImageNet pretrained ResNet18; no previous FER2013 checkpoint loaded',
        'classifier_hidden_layers': classifier_hidden_layers,
        'classifier_dropout': classifier_dropout,
        'classifier_shape': '512 -> 256 -> 7',
        'unfreeze_from': unfreeze_from,
        'optimizer_name': optimizer_name,
        'epochs': epochs,
        'lr': lr,
        'weight_decay': weight_decay,
        'use_class_weights': use_class_weights,
        'weighted_sampler': weighted_sampler,
        'loss_name': loss_name,
        'focal_gamma': focal_gamma,
        'selection_rule': 'highest validation macro F1; lower validation loss tie-breaker',
        'train_transform': '224x224 grayscale replicated to 3 channels, augment=True, ImageNet normalization',
        'eval_transform': '224x224 grayscale replicated to 3 channels, augment=False, ImageNet normalization',
        'checkpoint': checkpoint_path,
    })
    return finalize_model_run(model, loaders, history, name, checkpoint_path)


### Feature Extraction: Classifier Only

Train only the new dropout classifier head. This is the clean feature-extraction baseline.


In [ ]:
history_resnet_fc, report_resnet_fc, confusions_resnet_fc = run_resnet18_fresh_experiment(
    name='resnet18_feature_extract_fc',
    unfreeze_from='fc',
    epochs=15,
)


### Unfreeze Layer4

Fresh ImageNet start again, but train `layer4` plus the classifier head.


In [ ]:
history_resnet_layer4, report_resnet_layer4, confusions_resnet_layer4 = run_resnet18_fresh_experiment(
    name='resnet18_unfreeze_layer4',
    unfreeze_from='layer4',
    epochs=15,
)


### Unfreeze Layer3

Fresh ImageNet start again, now training `layer3`, `layer4`, and the classifier head.


In [ ]:
history_resnet_layer3, report_resnet_layer3, confusions_resnet_layer3 = run_resnet18_fresh_experiment(
    name='resnet18_unfreeze_layer3',
    unfreeze_from='layer3',
    epochs=15,
)


### Unfreeze Layer2

Fresh ImageNet start again, now training from `layer2` through the classifier head. Use this only after inspecting the shallower runs, because it has a larger trainable surface.


In [ ]:
history_resnet_layer2, report_resnet_layer2, confusions_resnet_layer2 = run_resnet18_fresh_experiment(
    name='resnet18_unfreeze_layer2',
    unfreeze_from='layer2',
    epochs=15,
)


## Controlled Imbalance and Loss Tuning

After the depth search, set `BEST_RESNET18_UNFREEZE_FROM` to the depth you want to continue testing. These runs still start fresh from ImageNet weights; they do not load the best checkpoint from the depth-search runs.


In [ ]:
BEST_RESNET18_UNFREEZE_FROM = 'layer4'


### Class Weights


In [ ]:
history_resnet_class_weights, report_resnet_class_weights, confusions_resnet_class_weights = run_resnet18_fresh_experiment(
    name=f'resnet18_{BEST_RESNET18_UNFREEZE_FROM}_class_weights',
    unfreeze_from=BEST_RESNET18_UNFREEZE_FROM,
    epochs=15,
    use_class_weights=True,
    weighted_sampler=False,
    loss_name='cross_entropy',
)


### WeightedRandomSampler


In [ ]:
history_resnet_weighted_sampler, report_resnet_weighted_sampler, confusions_resnet_weighted_sampler = run_resnet18_fresh_experiment(
    name=f'resnet18_{BEST_RESNET18_UNFREEZE_FROM}_weighted_sampler',
    unfreeze_from=BEST_RESNET18_UNFREEZE_FROM,
    epochs=15,
    use_class_weights=False,
    weighted_sampler=True,
    loss_name='cross_entropy',
)


### Focal Loss


In [ ]:
history_resnet_focal, report_resnet_focal, confusions_resnet_focal = run_resnet18_fresh_experiment(
    name=f'resnet18_{BEST_RESNET18_UNFREEZE_FROM}_focal',
    unfreeze_from=BEST_RESNET18_UNFREEZE_FROM,
    epochs=15,
    use_class_weights=False,
    weighted_sampler=False,
    loss_name='focal',
    focal_gamma=1.0,
)


### Class Weights + Focal Loss


In [ ]:
history_resnet_class_weights_focal, report_resnet_class_weights_focal, confusions_resnet_class_weights_focal = run_resnet18_fresh_experiment(
    name=f'resnet18_{BEST_RESNET18_UNFREEZE_FROM}_class_weights_focal',
    unfreeze_from=BEST_RESNET18_UNFREEZE_FROM,
    epochs=15,
    use_class_weights=True,
    weighted_sampler=False,
    loss_name='focal',
    focal_gamma=1.0,
)


### WeightedRandomSampler + Focal Loss


In [ ]:
history_resnet_weighted_sampler_focal, report_resnet_weighted_sampler_focal, confusions_resnet_weighted_sampler_focal = run_resnet18_fresh_experiment(
    name=f'resnet18_{BEST_RESNET18_UNFREEZE_FROM}_weighted_sampler_focal',
    unfreeze_from=BEST_RESNET18_UNFREEZE_FROM,
    epochs=15,
    use_class_weights=False,
    weighted_sampler=True,
    loss_name='focal',
    focal_gamma=1.0,
)


### WeightedRandomSampler + Focal Loss + SGD Plateau

This mirrors the strongest self-defined CNN training recipe more closely: weighted sampling, softened focal loss, SGD momentum, gradient clipping, and validation-loss plateau learning-rate reduction.


In [ ]:
history_resnet_sgd, report_resnet_sgd, confusions_resnet_sgd = run_resnet18_fresh_experiment(
    name=f'resnet18_{BEST_RESNET18_UNFREEZE_FROM}_weighted_sampler_focal_sgd_plateau_lr',
    unfreeze_from=BEST_RESNET18_UNFREEZE_FROM,
    epochs=40,
    lr=0.01,
    optimizer_name='sgd_plateau',
    weight_decay=5e-4,
    use_class_weights=False,
    weighted_sampler=True,
    loss_name='focal',
    focal_gamma=1.0,
    lr_decay_rate=0.1,
    lr_plateau_patience=5,
    lr_plateau_threshold=1e-3,
    min_lr=1e-6,
)


## Merge ResNet18 Results

Run this after finishing several ResNet18 cells. It reads saved metric files and creates one comparison table without retraining.


In [ ]:
metrics_dir = RESULTS_DIR / 'metrics'
summary_names = list(DEPTH_SEARCH_NAMES)
summary_names += [
    f'resnet18_{BEST_RESNET18_UNFREEZE_FROM}_class_weights',
    f'resnet18_{BEST_RESNET18_UNFREEZE_FROM}_weighted_sampler',
    f'resnet18_{BEST_RESNET18_UNFREEZE_FROM}_focal',
    f'resnet18_{BEST_RESNET18_UNFREEZE_FROM}_class_weights_focal',
    f'resnet18_{BEST_RESNET18_UNFREEZE_FROM}_weighted_sampler_focal',
    f'resnet18_{BEST_RESNET18_UNFREEZE_FROM}_weighted_sampler_focal_sgd_plateau_lr',
]

existing_resnet_reports = [
    path.name.removesuffix('_classification_report.csv')
    for path in metrics_dir.glob('resnet18*_classification_report.csv')
    if 'rafdb' not in path.name
]
summary_names = list(dict.fromkeys(summary_names + existing_resnet_reports))

summary_rows = {}
for name in summary_names:
    report_path = metrics_dir / f'{name}_classification_report.csv'
    history_path = metrics_dir / f'{name}_history.csv'
    confusions_path = metrics_dir / f'{name}_top_confusions.csv'
    config_path = metrics_dir / f'{name}_config.json'
    if not report_path.exists() or not history_path.exists():
        continue

    report = pd.read_csv(report_path, index_col=0)
    history = pd.read_csv(history_path)
    confusions = pd.read_csv(confusions_path) if confusions_path.exists() else pd.DataFrame()
    config = json.loads(config_path.read_text()) if config_path.exists() else {}
    best_row = best_history_row(history)
    summary_rows[name] = {
        'unfreeze_from': config.get('unfreeze_from', 'unknown'),
        'optimizer': config.get('optimizer_name', 'unknown'),
        'loss': config.get('loss_name', 'unknown'),
        'weighted_sampler': config.get('weighted_sampler', 'unknown'),
        'class_weights': config.get('use_class_weights', 'unknown'),
        'best_epoch': int(best_row['epoch']),
        'best_val_loss': float(best_row['val_loss']),
        'best_val_macro_f1': float(best_row['val_macro_f1']),
        'test_macro_f1': float(report.loc['macro avg', 'f1-score']),
        'test_weighted_f1': float(report.loc['weighted avg', 'f1-score']),
        'top_confusion': 'none' if confusions.empty else f"{confusions.iloc[0]['true_emotion']} -> {confusions.iloc[0]['predicted_emotion']}",
    }

resnet18_summary = pd.DataFrame(summary_rows).T.sort_values('test_macro_f1', ascending=False)
resnet18_summary.to_csv(metrics_dir / 'resnet18_experiment_summary.csv')
resnet18_summary
